# ACRE Engine - Clustering Implementation & Optimization
## Notebook 5: PCA-Enhanced K-Means Clustering

**Author:** Daniella Tahchi  
**Date:** 2024  
**Purpose:** Implement adaptive clustering strategy with PCA optimization to maximize silhouette score  
**Input:** Feature matrix from Notebook 03, characterization results from Notebook 04  
**Output:** Optimal clustering model with best silhouette score

---

## Objectives

1. **Apply PCA dimensionality reduction** with multiple configurations
2. **Grid search optimization** across PCA dimensions and cluster counts (k=3 to 15)
3. **Maximize silhouette score** through systematic evaluation
4. **Train final production model** using optimal parameters
5. **Save all clustering artifacts** for the ACRE recommendation algorithm

---

## Table of Contents

1. [Setup & Configuration](#setup)
2. [Load Feature Artifacts](#load)
3. [PCA Grid Search Optimization](#pca-grid)
4. [Final Model Training](#final-model)
5. [Cluster Analysis & Profiling](#analysis)
6. [Save Clustering Artifacts](#save)

---

## 1. Setup & Configuration 

In [2]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Scikit-learn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score

import warnings
from pathlib import Path
from datetime import datetime
import joblib
import json
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Visualization settings
sns.set_style('whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("="*80)
print("ACRE ENGINE - CLUSTERING IMPLEMENTATION & OPTIMIZATION")
print("="*80)
print(f"\nNotebook executed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n✓ Libraries loaded successfully\n")

ACRE ENGINE - CLUSTERING IMPLEMENTATION & OPTIMIZATION

Notebook executed: 2026-04-17 16:58:10

✓ Libraries loaded successfully



### Configuration Parameters

In [3]:
# Configuration
CONFIG = {
    # Paths
    'input_dir': Path('../artifacts/feature-engineering'),
    'char_dir': Path('../artifacts/data-characterization'),
    'output_dir': Path('../artifacts/clustering'),
    
    # Clustering parameters
    'k_range': [5, 8, 10, 12, 15, 18, 20, 25, 30, 35, 40],  # Test k from 5 to 20
    
    # PCA configurations to test (optimize for silhouette score)
    'pca_configs': {
        'PCA_95_variance': 0.95,  
        'PCA_90_variance': 0.90, 
        'PCA_50_components': 50,  
        'PCA_30_components': 30
    },
    
    'random_state': 42
}

# Create output directory
CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()}, indent=2))
print("\n✓ Setup complete. Directories initialized.")

Configuration:
{
  "input_dir": "..\\artifacts\\feature-engineering",
  "char_dir": "..\\artifacts\\data-characterization",
  "output_dir": "..\\artifacts\\clustering",
  "k_range": [
    5,
    8,
    10,
    12,
    15,
    18,
    20,
    25,
    30,
    35,
    40
  ],
  "pca_configs": {
    "PCA_95_variance": 0.95,
    "PCA_90_variance": 0.9,
    "PCA_50_components": 50,
    "PCA_30_components": 30
  },
  "random_state": 42
}

✓ Setup complete. Directories initialized.


---
## 2. Load Feature Artifacts 

In [4]:
print("="*80)
print("LOADING DATA ARTIFACTS")
print("="*80)

# Load feature matrix
feature_matrix_path = CONFIG['input_dir'] / 'feature_matrix.npy'
feature_matrix = np.load(feature_matrix_path)

# Load movie index
movie_index_path = CONFIG['input_dir'] / 'movie_index.csv'
movie_index = pd.read_csv(movie_index_path)

n_samples, n_features = feature_matrix.shape

print(f"\n✓ Feature matrix loaded: {n_samples:,} movies × {n_features} features")
print(f"✓ Movie index loaded: {len(movie_index):,} movies")

# Quick sanity check
assert len(movie_index) == n_samples, "Mismatch between feature matrix and movie index!"

print("\n✓ Data quality verified")
print("="*80)

LOADING DATA ARTIFACTS

✓ Feature matrix loaded: 27,777 movies × 172 features
✓ Movie index loaded: 27,777 movies

✓ Data quality verified


---
## 3. PCA Grid Search Optimization 

### Strategy: Maximize Silhouette Score

High-dimensional sparse data often suffers from the "curse of dimensionality," where distance metrics become less meaningful, artificially deflating silhouette scores.

**Grid Search Parameters:**
- **PCA Dimensions**: Test 50 components and 20 components
- **Cluster Count (k)**: Iterate from 3 to 15
- **Algorithm**: K-Means (recommended by Notebook 04 for uniform density data)

**Objective**: Find the configuration that maximizes the silhouette score.

In [5]:
print("="*80)
print("GRID SEARCH: PCA CONFIGURATIONS × K-MEANS (k=5 to 20)")
print("="*80)

results = []
pca_models = {}
reduced_matrices = {}

# ──────────────────────────────────────────
# Step 1: Pre-compute PCA transformations
# ──────────────────────────────────────────

print("\n[Step 1] Computing PCA transformations...\n")

for name, param in CONFIG['pca_configs'].items():
    pca = PCA(n_components=param, random_state=CONFIG['random_state'])
    reduced_matrix = pca.fit_transform(feature_matrix)
    
    pca_models[name] = pca
    reduced_matrices[name] = reduced_matrix
    
    actual_comps = pca.n_components_
    var_explained = sum(pca.explained_variance_ratio_) * 100
    
    print(f" • {name}: {actual_comps} components retained ({var_explained:.1f}% variance explained)")

# ──────────────────────────────────────────
# Step 2: Grid search across PCA configs and k values
# ──────────────────────────────────────────

print("\n[Step 2] Evaluating K-Means clustering across configurations...\n")

for pca_name, matrix in reduced_matrices.items():
    print(f"Evaluating {pca_name}...")
    
    for k in tqdm(CONFIG['k_range'], desc=f"Testing k"):
        kmeans = KMeans(n_clusters=k, random_state=CONFIG['random_state'], n_init='auto')
        labels = kmeans.fit_predict(matrix)
        
        # Calculate metrics
        sil_score = silhouette_score(matrix, labels)
        inertia = kmeans.inertia_
        
        results.append({
            'PCA_Config': pca_name,
            'k': k,
            'Silhouette_Score': sil_score,
            'Inertia': inertia
        })

results_df = pd.DataFrame(results)

# ──────────────────────────────────────────
# Step 3: Identify the absolute best configuration
# ──────────────────────────────────────────

best_run = results_df.loc[results_df['Silhouette_Score'].idxmax()]
best_pca_config = best_run['PCA_Config']
best_k = int(best_run['k'])
best_sil = best_run['Silhouette_Score']

print(f"\n{'='*80}")
print("🏆 OPTIMAL CONFIGURATION DISCOVERED")
print(f"{'='*80}")
print(f" • Dimensionality Reduction : {best_pca_config}")
print(f" • Optimal Clusters (k)     : {best_k}")
print(f" • Max Silhouette Score     : {best_sil:.4f}")
print(f"{'='*80}")

GRID SEARCH: PCA CONFIGURATIONS × K-MEANS (k=5 to 20)

[Step 1] Computing PCA transformations...

 • PCA_95_variance: 138 components retained (95.2% variance explained)
 • PCA_90_variance: 112 components retained (90.2% variance explained)
 • PCA_50_components: 50 components retained (72.2% variance explained)
 • PCA_30_components: 30 components retained (62.4% variance explained)

[Step 2] Evaluating K-Means clustering across configurations...

Evaluating PCA_95_variance...


Testing k: 100%|██████████| 11/11 [01:29<00:00,  8.17s/it]


Evaluating PCA_90_variance...


Testing k: 100%|██████████| 11/11 [01:21<00:00,  7.44s/it]


Evaluating PCA_50_components...


Testing k: 100%|██████████| 11/11 [01:21<00:00,  7.40s/it]


Evaluating PCA_30_components...


Testing k: 100%|██████████| 11/11 [01:19<00:00,  7.21s/it]


🏆 OPTIMAL CONFIGURATION DISCOVERED
 • Dimensionality Reduction : PCA_30_components
 • Optimal Clusters (k)     : 40
 • Max Silhouette Score     : 0.2253


### Visualize Grid Search Results

In [6]:
# Silhouette score comparison across configurations
fig = px.line(
    results_df,
    x='k',
    y='Silhouette_Score',
    color='PCA_Config',
    markers=True,
    title='Clustering Quality: Silhouette Score by PCA Configuration & Cluster Count',
    labels={'k': 'Number of Clusters (k)', 'Silhouette_Score': 'Silhouette Score (↑ better)'}
)

# Highlight the maximum point
fig.add_annotation(
    x=best_k,
    y=best_sil,
    text=f"Optimal: k={best_k}<br>Score={best_sil:.3f}",
    showarrow=True,
    arrowhead=1,
    arrowsize=2,
    arrowcolor='red',
    font=dict(color='white'),
    bgcolor='red',
    ax=0,
    ay=-40
)

fig.update_layout(height=500, legend_title_text='PCA Configuration')
fig.show()

In [7]:
# ==================== CLUSTER QUALITY VALIDATION ====================

print("="*80)
print("LOADING DATA AND RECREATING WINNING MODEL")
print("="*80)

import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from pathlib import Path

# 1. Load original movie data
df = pd.read_csv('../Artifacts/preprocessing/cleaned_movies.csv')
print(f"\n✓ Loaded {len(df):,} movies")

# 2. Load feature matrix
feature_matrix = np.load('../Artifacts/feature-engineering/feature_matrix.npy')
print(f"✓ Feature matrix: {feature_matrix.shape}")

# 3. Verify alignment
assert len(df) == feature_matrix.shape[0], "Row mismatch!"
print(f"✓ Data aligned: {len(df)} rows in both\n")

# 4. Recreate winning model
print("Recreating optimal model (PCA_30 + k=40)...")
pca_30 = PCA(n_components=30, random_state=42)
X_reduced = pca_30.fit_transform(feature_matrix)

kmeans_40 = KMeans(n_clusters=40, random_state=42, n_init=10)
labels = kmeans_40.fit_predict(X_reduced)

# 5. Add cluster assignments to df
df['cluster'] = labels
print(f"✓ Assigned {len(set(labels))} clusters to movies\n")

# 6. VALIDATION: Manual cluster inspection
print("="*80)
print("CLUSTER QUALITY CHECK")
print("="*80)

for cluster_id in range(10):  # Check first 10 clusters
    print(f"\n{'─'*80}")
    print(f"CLUSTER {cluster_id} - Random Sample (15 movies):")
    print(f"{'─'*80}")
    
    # Get all movies in this cluster
    cluster_movies = df[df['cluster'] == cluster_id]
    n_movies_in_cluster = len(cluster_movies)
    
    print(f"Total movies in cluster: {n_movies_in_cluster}")
    
    # Sample movies (or all if fewer than 15)
    sample_size = min(15, n_movies_in_cluster)
    cluster_sample = cluster_movies.sample(sample_size, random_state=42)
    
    # Display key columns
    print("\n" + cluster_sample[['title', 'genres', 'vote_average']].to_string(index=False))
    
    # Genre distribution
    all_genres = cluster_movies['genres'].str.split(', ').explode()
    genre_dist = all_genres.value_counts()
    
    print(f"\nTop 3 genres in cluster:")
    for genre, count in genre_dist.head(3).items():
        pct = 100 * count / n_movies_in_cluster
        print(f"  - {genre}: {count} ({pct:.1f}%)")
    
    # Coherence check
    print(f"\n{'─'*80}")
    coherence = input(f"Does Cluster {cluster_id} make sense? (y/n/s to skip): ")
    
    if coherence.lower() == 'n':
        print(f"⚠️ WARNING: Cluster {cluster_id} flagged as problematic!")
    elif coherence.lower() == 'y':
        print(f"✓ Cluster {cluster_id} validated")
    elif coherence.lower() == 's':
        print("Skipping remaining clusters...")
        break

print("\n" + "="*80)
print("VALIDATION COMPLETE")
print("="*80)

LOADING DATA AND RECREATING WINNING MODEL

✓ Loaded 27,777 movies
✓ Feature matrix: (27777, 172)
✓ Data aligned: 27777 rows in both

Recreating optimal model (PCA_30 + k=40)...
✓ Assigned 40 clusters to movies

CLUSTER QUALITY CHECK

────────────────────────────────────────────────────────────────────────────────
CLUSTER 0 - Random Sample (15 movies):
────────────────────────────────────────────────────────────────────────────────
Total movies in cluster: 2770

                      title                 genres  vote_average
           The Beachnickers                 Comedy         6.900
              It's a Fairy!        Comedy, Fantasy         5.700
         Return to Mayberry       TV Movie, Comedy         6.117
                 Bagnomaria                 Comedy         5.879
             The Babymakers                 Comedy         5.149
             Doctor Detroit                 Comedy         5.078
              The Promotion                 Comedy         5.302
              

In [8]:
# Elbow curve for the best PCA configuration
best_pca_results = results_df[results_df['PCA_Config'] == best_pca_config]

fig2 = px.line(
    best_pca_results,
    x='k',
    y='Inertia',
    markers=True,
    title=f'Elbow Method (Inertia) for {best_pca_config}',
    labels={'k': 'Number of Clusters (k)', 'Inertia': 'Inertia (↓ better)'}
)

fig2.update_traces(line_color='#2E86AB')
fig2.update_layout(height=400)
fig2.show()

---
## 4. Final Model Training

Using the optimal parameters identified in the grid search:
- **PCA Configuration**: `best_pca_config`
- **Optimal k**: `best_k`

We will now:
1. Train the final production K-Means model
2. Extract cluster centroids (required for ACRE algorithm)
3. Assign every movie to its cluster

In [9]:
print("="*80)
print("TRAINING FINAL MODEL")
print("="*80)

# Retrieve the best PCA transformation
final_pca_model = pca_models[best_pca_config]
final_feature_matrix = reduced_matrices[best_pca_config]

# Train Final K-Means
final_kmeans = KMeans(
    n_clusters=best_k,
    random_state=CONFIG['random_state'],
    n_init=10  # Explicit n_init for production model stability
)

print(f"\nFitting final K-Means model (k={best_k}) on {final_feature_matrix.shape[1]} dimensions...")

movie_labels = final_kmeans.fit_predict(final_feature_matrix)
cluster_centroids = final_kmeans.cluster_centers_

print("✓ Model fitted successfully.")
print(f"✓ Extracted {len(cluster_centroids)} cluster centroids.")

# Map labels back to the movie index
movie_index['cluster'] = movie_labels

print("✓ Cluster assignments completed.")
print("="*80)

TRAINING FINAL MODEL

Fitting final K-Means model (k=40) on 30 dimensions...
✓ Model fitted successfully.
✓ Extracted 40 cluster centroids.
✓ Cluster assignments completed.


---
## 5. Cluster Analysis & Profiling

### Cluster Distribution Analysis

In [10]:
print("="*80)
print("CLUSTER DISTRIBUTION ANALYSIS")
print("="*80)

cluster_counts = movie_index['cluster'].value_counts().reset_index()
cluster_counts.columns = ['Cluster', 'Movie Count']
cluster_counts['Percentage'] = (cluster_counts['Movie Count'] / len(movie_index)) * 100
cluster_counts = cluster_counts.sort_values('Cluster')

print("\nCluster Distribution:\n")
print(cluster_counts.to_string(index=False))

print(f"\nCluster Size Statistics:")
print(f" • Mean size: {cluster_counts['Movie Count'].mean():.0f} movies")
print(f" • Std dev: {cluster_counts['Movie Count'].std():.0f}")
print(f" • Min size: {cluster_counts['Movie Count'].min()} movies")
print(f" • Max size: {cluster_counts['Movie Count'].max()} movies")

print("="*80)

CLUSTER DISTRIBUTION ANALYSIS

Cluster Distribution:

 Cluster  Movie Count  Percentage
       0         2770    9.972279
       1          291    1.047629
       2          751    2.703676
       3          431    1.551643
       4         1325    4.770134
       5          584    2.102459
       6          510    1.836051
       7          914    3.290492
       8         1165    4.194117
       9          421    1.515642
      10         2364    8.510638
      11          280    1.008028
      12          992    3.571300
      13          722    2.599273
      14          284    1.022429
      15          568    2.044857
      16          284    1.022429
      17         1539    5.540555
      18          213    0.766821
      19          348    1.252835
      20          764    2.750477
      21          615    2.214062
      22          309    1.112431
      23         1238    4.456925
      24          623    2.242863
      25          570    2.052057
      26          361    1.2

### Cluster Quality Metrics

In [12]:
# Compute final quality metrics
final_silhouette = silhouette_score(final_feature_matrix, movie_labels)
final_calinski = calinski_harabasz_score(final_feature_matrix, movie_labels)
final_inertia = final_kmeans.inertia_

print("\nFinal Model Quality Metrics:\n")
print(f" • Silhouette Score       : {final_silhouette:.4f}")
print(f" • Calinski-Harabasz Score: {final_calinski:.2f}")
print(f" • Inertia                : {final_inertia:.2f}")
print(f" • Number of Clusters     : {best_k}")
print(f" • PCA Components Used    : {final_pca_model.n_components_}")


Final Model Quality Metrics:

 • Silhouette Score       : 0.2321
 • Calinski-Harabasz Score: 1114.46
 • Inertia                : 24938.45
 • Number of Clusters     : 40
 • PCA Components Used    : 30


---
## 6. Save Clustering Artifacts

As per **Document 1 (Section 5.6 Clustering Outputs)**, we must save the following artifacts for the ACRE recommendation API:

1. `clustering_model.joblib` - The fitted K-Means estimator
2. `pca_model.joblib` - The fitted PCA transformer
3. `cluster_centroids.npy` - The spatial centers of taste groups
4. `cluster_assignments.csv` - Movie IDs mapped to cluster labels
5. `reduction_method.txt` - Text flag signaling the active reduction method
6. `clustering_report.json` - Metadata and performance metrics

In [13]:
print("="*80)
print("SAVING CLUSTERING ARTIFACTS")
print("="*80)

out_dir = CONFIG['output_dir']

# ──────────────────────────────────────────
# 1. Save Clustering Model
# ──────────────────────────────────────────

print("\n[1] Saving clustering model...")
kmeans_path = out_dir / 'clustering_model.joblib'
joblib.dump(final_kmeans, kmeans_path)
print(f" ✓ Saved: {kmeans_path.name}")

# ──────────────────────────────────────────
# 2. Save PCA Model
# ──────────────────────────────────────────

print("\n[2] Saving PCA model...")
pca_path = out_dir / 'pca_model.joblib'
joblib.dump(final_pca_model, pca_path)
print(f" ✓ Saved: {pca_path.name}")

# ──────────────────────────────────────────
# 3. Save Centroids
# ──────────────────────────────────────────

print("\n[3] Saving cluster centroids...")
centroids_path = out_dir / 'cluster_centroids.npy'
np.save(centroids_path, cluster_centroids)
print(f" ✓ Saved: {centroids_path.name} (Shape: {cluster_centroids.shape})")

# ──────────────────────────────────────────
# 4. Save Cluster Assignments
# ──────────────────────────────────────────

print("\n[4] Saving cluster assignments...")
assignments_path = out_dir / 'cluster_assignments.csv'
movie_index.to_csv(assignments_path, index=False, encoding='utf-8')
print(f" ✓ Saved: {assignments_path.name}")

print("\n[5] Saving reduction method flag...")
reduction_path = out_dir / 'reduction_method.txt'
with open(reduction_path, 'w', encoding='utf-8') as f:
    f.write('pca')
print(f" ✓ Saved: {reduction_path.name}")

# ──────────────────────────────────────────
# 5. Save Clustering Report
# ──────────────────────────────────────────

print("\n[6] Saving clustering metadata report...")

report = {
    'timestamp': datetime.now().isoformat(),
    'algorithm': 'K-Means',
    'optimal_k': int(best_k),
    'pca_configuration': best_pca_config,
    'pca_components_used': int(final_pca_model.n_components_),
    'variance_explained': float(sum(final_pca_model.explained_variance_ratio_)),
    'final_silhouette_score': float(final_silhouette),
    'final_inertia': float(final_inertia),
    'calinski_harabasz_score': float(final_calinski),
    'n_movies_clustered': int(len(movie_index)),
    'original_features': int(n_features),
    'reduced_features': int(final_pca_model.n_components_)
}

report_path = out_dir / 'clustering_report.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=4)
print(f" ✓ Saved: {report_path.name}")

print(f"\n{'='*80}")
print("✅ CLUSTERING PIPELINE COMPLETE")
print(f"{'='*80}")
print(f"\nOutput directory: {out_dir.absolute()}")
print("\nArtifacts created:")
print(f" • clustering_model.joblib")
print(f" • pca_model.joblib")
print(f" • cluster_centroids.npy")
print(f" • cluster_assignments.csv")
print(f" • clustering_report.json")
print(f"\n✓ Ready for ACRE Recommendation Algorithm (Notebook 06)")
print("="*80)

SAVING CLUSTERING ARTIFACTS

[1] Saving clustering model...
 ✓ Saved: clustering_model.joblib

[2] Saving PCA model...
 ✓ Saved: pca_model.joblib

[3] Saving cluster centroids...
 ✓ Saved: cluster_centroids.npy (Shape: (40, 30))

[4] Saving cluster assignments...
 ✓ Saved: cluster_assignments.csv

[5] Saving reduction method flag...
 ✓ Saved: reduction_method.txt

[6] Saving clustering metadata report...
 ✓ Saved: clustering_report.json

✅ CLUSTERING PIPELINE COMPLETE

Output directory: c:\Users\user\Documents\University\Projects\ACRE-FUSE\Notebooks-ACRE\..\artifacts\clustering

Artifacts created:
 • clustering_model.joblib
 • pca_model.joblib
 • cluster_centroids.npy
 • cluster_assignments.csv
 • clustering_report.json

✓ Ready for ACRE Recommendation Algorithm (Notebook 06)


---

## Summary

### Clustering Pipeline Complete

✅ **Grid Search Completed**: Tested multiple PCA configurations and cluster counts  
✅ **Optimal Configuration Found**: Maximized silhouette score through systematic evaluation  
✅ **Production Model Trained**: Final K-Means model with optimal parameters  
✅ **Cluster Assignments**: All movies assigned to taste clusters  
✅ **Artifacts Saved**: All required files for ACRE recommendation algorithm

### Key Results

- **Algorithm**: K-Means (data-driven selection from Notebook 04)
- **Optimal k**: {best_k} clusters
- **PCA Configuration**: {best_pca_config}
- **Final Silhouette Score**: {best_sil:.4f}
- **Dimensionality Reduction**: {n_features} → {final_pca_model.n_components_} features

### Next Steps

→ **Notebook 06** - ACRE Recommendation Algorithm Implementation  
→ **Notebook 07** - Baseline Comparison & User Study Setup

---

**End of Notebook 05**